# Greedy OED: comparaison A / D / C / Random

Notebook de comparaison des 4 strategies (A-opt, D-opt, C-opt, Random)
inspire de `tutorials/examples/boed/run_greedy.py`, avec des tailles plus modestes pour
un lancement plus rapide.

Si vous voulez plus de fidelite, augmentez `N`, `n_steps` et `n_budget` ci-dessous.


In [2]:
import numpy as np
import numpy.linalg as la
import matplotlib.pyplot as plt
import sys
from pathlib import Path


# Import bootstrap compatible notebook/script and mixed environments

def _find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        # Case 1: running from inside pyBOED
        if (p / "boed" / "__init__.py").is_file():
            return p
        # Case 2: running from parent directory containing pyBOED/
        if (p / "pyBOED" / "boed" / "__init__.py").is_file():
            return p / "pyBOED"
    return cwd

PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# If another installed package named `boed` was already imported, clear it.
for name in list(sys.modules):
    if name == "boed" or name.startswith("boed."):
        del sys.modules[name]

print(f"Using PROJECT_ROOT={PROJECT_ROOT}")


from boed.priors.kernels import Gaussian
from boed.priors.gp_priors import GaussianProcessPrior
from boed.core.noise import NoiseModel
from boed.pde.advection_diffusion import AdvectionDiffusion1D_CN
from boed.inference import LinearGaussianModel
from boed.observations.sensors import SpaceTimeSensors
from boed.design.greedy import run_greedy_oed
from boed.design.criteria import DesignCriteria
from boed.core import make_u0

SEED = 42
rng = np.random.default_rng(SEED)

# Parametres (modestes pour rapidite)
N, dt, n_steps = 80, 0.01, 40
diffusivity, velocity = 0.01, 0.5
n_budget = 6

x_grid = np.linspace(0, 1, N)
model = AdvectionDiffusion1D_CN(N, dt, diffusivity=diffusivity, velocity=velocity)
u0 = make_u0(
    x_grid,
    "double_gaussian",
    centers=(0.2, 0.4),
    widths=(0.07, 0.058),
    amplitudes=(1.0, 0.5),
)
trajectory = model.evolve(u0, n_steps)

kernel = Gaussian(length_scale=0.1, sigma=1.0)
prior_process = GaussianProcessPrior(kernel, nx=N)
Sigma_prior, mu_prior = prior_process.Sigma, prior_process.mu
noise = NoiseModel(sigma_noise=0.01)

# Candidats
candidates_x = np.linspace(5, N - 5, 16, dtype=int)
candidates_t = np.linspace(0, n_steps, 10, dtype=int)

# QoI pour le critere C (moyenne zone gauche)
L_left = np.zeros(N)
L_left[N // 4 : N // 2] = 1.0 / (N // 4)

# Pre-calcul de l'operateur de trajectoire (u0 -> u(t))
M = model.get_transition_matrix()
Trajectory_Op = np.vstack([np.linalg.matrix_power(M, t) for t in range(n_steps + 1)])


Using PROJECT_ROOT=/home/mdoumbou/Documents/Biblio_thèse/pyBOED


In [3]:
# 1) Designs par critere
strategies = {}

print('Running A-opt...')
des_A, hist_A, _ = run_greedy_oed(model, Sigma_prior, noise, candidates_x, candidates_t, n_budget, 'A')
strategies['A-Opt'] = (des_A, hist_A)

print('Running D-opt...')
des_D, hist_D, _ = run_greedy_oed(model, Sigma_prior, noise, candidates_x, candidates_t, n_budget, 'D')
strategies['D-Opt'] = (des_D, hist_D)

print('Running C-opt...')
des_C, hist_C, _ = run_greedy_oed(model, Sigma_prior, noise, candidates_x, candidates_t, n_budget, 'C', L_qoi=L_left)
strategies['C-Opt'] = (des_C, hist_C)

print('Random baseline...')
rand_idx = rng.choice(len(candidates_x) * len(candidates_t), n_budget, replace=False)
des_rand = []
for idx in rand_idx:
    ti_idx = idx // len(candidates_x)
    xi_idx = idx % len(candidates_x)
    des_rand.append((int(candidates_x[xi_idx]), int(candidates_t[ti_idx])))
strategies['Random'] = (des_rand, None)


Running A-opt...
--- Optimisation OED (Critère A) ---
Étape 1/6 : x=56, t=40 | Score: 6.3108e+01
Étape 2/6 : x=33, t=40 | Score: 4.7247e+01
Étape 3/6 : x=75, t=26 | Score: 3.1791e+01
Étape 4/6 : x=28, t=40 | Score: 2.2314e+01
Étape 5/6 : x=75, t=40 | Score: 1.3626e+01
Étape 6/6 : x=75, t=4 | Score: 8.5056e+00
Running D-opt...
--- Optimisation OED (Critère D) ---
Étape 1/6 : x=14, t=0 | Score: -1.3814e+03
Étape 2/6 : x=51, t=0 | Score: -1.3907e+03
Étape 3/6 : x=75, t=0 | Score: -1.3999e+03
Étape 4/6 : x=33, t=0 | Score: -1.4091e+03
Étape 5/6 : x=65, t=0 | Score: -1.4180e+03
Étape 6/6 : x=5, t=0 | Score: -1.4269e+03
Running C-opt...
--- Optimisation OED (Critère C) ---
Étape 1/6 : x=42, t=31 | Score: 1.0400e-03
Étape 2/6 : x=9, t=0 | Score: 8.5519e-04
Étape 3/6 : x=33, t=8 | Score: 6.7597e-04
Étape 4/6 : x=47, t=0 | Score: 3.9163e-04
Étape 5/6 : x=14, t=4 | Score: 2.3018e-04
Étape 6/6 : x=51, t=8 | Score: 2.0959e-04
Random baseline...


In [ ]:
def evaluate_design(design):
    unique_pts = sorted(list(set(design)), key=lambda x: x[1])
    sensors = SpaceTimeSensors([p[0] for p in unique_pts], [p[1] for p in unique_pts], N)
    W = sensors.observation_operator(n_steps + 1)
    A_fwd = W @ Trajectory_Op

    y_obs = A_fwd @ u0 + rng.normal(0, noise.sigma, size=len(unique_pts))
    Sigma_eps = (noise.sigma ** 2) * np.eye(len(unique_pts))

    lgm = LinearGaussianModel(A_fwd, Sigma_eps, mu_prior, Sigma_prior)
    mu_post, Sigma_post = lgm.posterior(y_obs)

    eig = DesignCriteria.EIG(Sigma_post, Sigma_prior)
    err_global = la.norm(u0 - mu_post)
    qoi_true = float(L_left @ u0)
    qoi_est = float(L_left @ mu_post)
    err_qoi = abs(qoi_true - qoi_est)

    return {
        'mu': mu_post,
        'Sigma': Sigma_post,
        'sensors': sensors,
        'eig': eig,
        'err_global': err_global,
        'err_qoi': err_qoi,
    }

results = {}
for name, (design, _hist) in strategies.items():
    results[name] = evaluate_design(design)
    print(f"{name}: EIG={results[name]['eig']:.2f} | ErrGlobal={results[name]['err_global']:.3f} | ErrQoI={results[name]['err_qoi']:.4f}")


In [ ]:
# Figure 1: designs spatio-temporels (temps en abscisse)
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
for i, name in enumerate(results.keys()):
    ax = axes[i]

    # trajectory shape: (n_steps+1, N) -> x-axis = time, y-axis = space index
    ax.imshow(
        trajectory,
        aspect='auto',
        origin='lower',
        extent=[0, n_steps, 0, N],
        cmap='viridis',
    )

    sx = results[name]['sensors'].x_idx  # spatial indices
    st = results[name]['sensors'].t_idx  # time indices
    ax.scatter(st, sx, c='r', marker='x', s=80, linewidth=2)

    ax.set_title(f"{name} (EIG: {results[name]['eig']:.1f})")
    ax.set_xlabel('t')
    ax.set_ylabel('x')

plt.tight_layout()
plt.show()


In [ ]:
# Figure 2: reconstruction comparee
plt.figure(figsize=(12, 5))
plt.plot(x_grid, u0, 'k-', lw=2, label='True u0')
for name, style in [('A-Opt','--'), ('D-Opt','-.'), ('C-Opt',':'), ('Random','-')]:
    plt.plot(x_grid, results[name]['mu'], style, lw=2, label=name)
plt.axvspan(x_grid[N//4], x_grid[N//2], color='green', alpha=0.1, label='QoI zone')
plt.title('Posterior mean (compare A/D/C/Random)')
plt.xlabel('x')
plt.ylabel('u')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
# Figure 3: spectre des valeurs propres
plt.figure(figsize=(10, 4))
for name in results:
    eigvals = np.linalg.eigvalsh(results[name]['Sigma'])
    plt.semilogy(np.sort(eigvals)[::-1], label=name)
plt.semilogy(np.sort(np.linalg.eigvalsh(Sigma_prior))[::-1], 'k--', label='Prior')
plt.title('Spectre des variances (posterior)')
plt.xlabel('Mode')
plt.ylabel('Eigenvalue')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
# Figure 4: comparaison des 4 strategies (EIG / erreurs)
names = list(results.keys())
eig_vals = [results[n]['eig'] for n in names]
err_global = [results[n]['err_global'] for n in names]
err_qoi = [results[n]['err_qoi'] for n in names]

x = np.arange(len(names))
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].bar(x, eig_vals)
axes[0].set_title('EIG (nats)')
axes[0].set_ylabel('higher is better')

axes[1].bar(x, err_global)
axes[1].set_title('Global error')
axes[1].set_ylabel('lower is better')

axes[2].bar(x, err_qoi)
axes[2].set_title('QoI error')
axes[2].set_ylabel('lower is better')

for ax in axes:
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=20)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()
